In [ ]:
import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import matplotlib.gridspec as gridspec

file_spei = os.path.join("PATH_TO_SPEI_INPUT", "SPEI_Rd111.xlsx")
file_smrz = os.path.join("PATH_TO_SMRZ_INPUT", "SMrz_Rd111.xlsx")
out_png = os.path.join("PATH_TO_OUTPUT", "Combined_SPEI_SMrz_Season_Rd111.png")

plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 11,
    "axes.linewidth": 0.6,
    "figure.facecolor": "white",
    "axes.facecolor": "white"
})

type_color_map = {
    'VPD': "#BDBDBD",
    'SRAD': "#E6E6E6",
    'LAI': "#BDBDBD",
    'CO2': "#E6E6E6",
    'PRE': "#BDBDBD",
    'TEM': "#E6E6E6"
}

RANK_FILL_RATIO = {
    1: 0.82,
    2: 0.63,
    3: 0.44,
}

COLOR_SPEI = "#23025c"
COLOR_SMRZ = "#00b2cb"
MARKER_SPEI = "o"
MARKER_SMRZ = "D"
BUBBLE_EDGE_COLOR = "white"
BUBBLE_EDGE_LW = 0.5
OFFSET_X = 0.0
OFFSET_Y = 0.0

cell_edge_color = "#E6E6E6"
cell_edge_lw = 0.6
group_line_color = "#BDBDBD"
group_line_lw = 1.2

climates = ['Temperate', 'Tropical', 'Boreal', 'Arid']
seasons = ['MAM', 'JJA', 'SON', 'DJF']
gap_rows = 1

metric_order = ("mean", "trend", "cv")

bar_height_ratio = 0.3
heat_height_ratio = 6.0
legend_ratio = 0.28
bar_group_width = 0.86
bar_gap = 0
bar_single_width = (bar_group_width - bar_gap) / 2
base_color = "#E9EEF2"
show_bar_grid = False

shift = -0.14
gap = -0.165

DIAMOND_MS_SCALE = 0.72


def format_label(label: str) -> str:
    s = str(label)
    s = s.replace("_Y_", " Y_")
    s = s.replace("CO2", "CO$_2$")
    s = s.replace("SMrz", "SM$_{rz}$")
    return s


def parse_var_name(v: str):
    s = str(v).strip()
    m = re.match(r"^(?P<driver>.+?)(?P<y>_Y)?_(?P<metric>Mean|Trend|CV)$",
                 s, flags=re.IGNORECASE)
    if not m:
        return s, False, None
    return m.group("driver"), m.group("y") is not None, m.group("metric").lower()


def get_var_prefix(var_name: str):
    driver, _, _ = parse_var_name(str(var_name))
    for p in type_color_map:
        if driver.startswith(p):
            return p
    return "OTHER"


def read_rank_excel(path):
    df = pd.read_excel(path, header=[0, 1])
    variables = df.iloc[:, 0].astype(str).values
    rows = []
    for i, var in enumerate(variables):
        for j in range(1, len(df.columns)):
            v = df.iloc[i, j]
            if pd.notna(v) and str(v).strip() != "":
                try:
                    rk = int(float(v))
                    if rk in (1, 2, 3):
                        rows.append({
                            "variable": str(var),
                            "climate": str(df.columns[j][0]).strip(),
                            "season": str(df.columns[j][1]).strip(),
                            "rank": rk
                        })
                except:
                    pass
    return pd.DataFrame(rows), variables


def build_y_layout(seasons, climates, gap_rows=1):
    y_pos, y_to_climate, season_centers = {}, {}, {}
    y = 0
    for si, s in enumerate(seasons):
        start_y = y
        for c in climates:
            y_pos[(s, c)] = y
            y_to_climate[y] = c
            y += 1
        season_centers[s] = (start_y + y - 1) / 2
        if si < len(seasons) - 1:
            for _ in range(gap_rows):
                y_to_climate[y] = None
                y += 1
    return y_pos, y_to_climate, season_centers, y


def sort_and_group_variables(vars_array, metric_order=("mean", "trend", "cv")):
    vars_list = [str(v) for v in vars_array]
    driver_order, seen = [], set()
    for v in vars_list:
        d, _, _ = parse_var_name(v)
        if d not in seen:
            seen.add(d)
            driver_order.append(d)
    metric_rank = {m: i for i, m in enumerate(metric_order)}

    def key(v):
        d, iy, m = parse_var_name(v)
        return (
            driver_order.index(d) if d in driver_order else 1e9,
            1 if iy else 0,
            metric_rank.get(m, 1e6),
            v
        )

    vs = sorted(vars_list, key=key)
    groups = []
    for d in driver_order:
        idxs = [i for i, v in enumerate(vs) if parse_var_name(v)[0] == d]
        if idxs:
            groups.append((d, min(idxs), max(idxs)))
    return vs, groups


def count_ranks(variables_sorted, climates, seasons, data_map, ranks=(1, 2, 3)):
    counts = {v: {rk: 0 for rk in ranks} for v in variables_sorted}
    for v in variables_sorted:
        for s in seasons:
            for c in climates:
                r = data_map.get((v, c, s))
                if r in ranks:
                    counts[v][r] += 1
    return counts


def compute_s_values(ax, fig, rank_fill_ratio):
    fig.canvas.draw()
    ax_width_inch = ax.get_position().width * fig.get_figwidth()
    ax_width_pts = ax_width_inch * 72.0
    xlim = ax.get_xlim()
    x_data_range = xlim[1] - xlim[0]
    cell_pts = ax_width_pts / x_data_range
    s_dict = {}
    for rk, ratio in rank_fill_ratio.items():
        d_pts = ratio * cell_pts
        s_dict[rk] = np.pi * (d_pts / 2) ** 2
    return s_dict, cell_pts


df_spei, variables_spei = read_rank_excel(file_spei)
df_smrz, variables_smrz = read_rank_excel(file_smrz)

if list(map(str, variables_spei)) != list(map(str, variables_smrz)):
    raise ValueError("Variable lists are not consistent between the two tables.")

variables_sorted, var_groups = sort_and_group_variables(
    variables_spei, metric_order=metric_order
)
nx = len(variables_sorted)

y_pos, y_to_climate, season_centers, ny = build_y_layout(
    seasons, climates, gap_rows=gap_rows
)

spei_map = {(r.variable, r.climate, r.season): int(r.rank) for r in df_spei.itertuples()}
smrz_map = {(r.variable, r.climate, r.season): int(r.rank) for r in df_smrz.itertuples()}

counts_spei = count_ranks(variables_sorted, climates, seasons, spei_map)
counts_smrz = count_ranks(variables_sorted, climates, seasons, smrz_map)

fig_w = max(9.0, nx * 0.28)
fig_h = max(5.8, ny * 0.42 + 1.5)

fig = plt.figure(figsize=(fig_w, fig_h), dpi=600)

gs = gridspec.GridSpec(
    3, 1,
    height_ratios=[legend_ratio, bar_height_ratio, heat_height_ratio],
    hspace=0.0
)
ax_legend = fig.add_subplot(gs[0])
ax_bar = fig.add_subplot(gs[1])
ax = fig.add_subplot(gs[2], sharex=ax_bar)

ax_legend.axis("off")

x_centers = np.arange(nx) + 0.5
x_left = x_centers - (bar_gap / 2 + bar_single_width / 2)
x_right = x_centers + (bar_gap / 2 + bar_single_width / 2)

totals_s = np.array([sum(counts_spei[v].values()) for v in variables_sorted])
totals_m = np.array([sum(counts_smrz[v].values()) for v in variables_sorted])
ymax_bar = int(max(totals_s.max(initial=0), totals_m.max(initial=0), 1))

ax_bar.bar(x_left, np.full(nx, ymax_bar), width=bar_single_width,
           color=base_color, edgecolor="none", zorder=0)
ax_bar.bar(x_right, np.full(nx, ymax_bar), width=bar_single_width,
           color=base_color, edgecolor="none", zorder=0)

rank_alphas = {1: 1.0, 2: 0.7, 3: 0.4}
bot_l = np.zeros(nx)
bot_r = np.zeros(nx)
for rk in [1, 2, 3]:
    vl = np.array([counts_spei[v][rk] for v in variables_sorted], dtype=float)
    vr = np.array([counts_smrz[v][rk] for v in variables_sorted], dtype=float)
    ax_bar.bar(x_left, vl, bottom=bot_l, width=bar_single_width,
               color=COLOR_SPEI, alpha=rank_alphas[rk], edgecolor="none", zorder=2)
    ax_bar.bar(x_right, vr, bottom=bot_r, width=bar_single_width,
               color=COLOR_SMRZ, alpha=rank_alphas[rk], edgecolor="none", zorder=2)
    bot_l += vl
    bot_r += vr

ax_bar.set_ylim(0, ymax_bar)
ax_bar.set_ylabel("Count", fontsize=11)
ax_bar.set_yticks([0, ymax_bar])
ax_bar.set_yticklabels(["0", str(ymax_bar)], fontsize=11)
ax_bar.tick_params(axis="x", which="both", bottom=False, labelbottom=False, length=0)
ax_bar.tick_params(axis="y", length=0)

for sp in ax_bar.spines.values():
    sp.set_visible(False)
if show_bar_grid:
    ax_bar.grid(axis="y", color="#BDBDBD", lw=0.6, alpha=0.6)
for _, sx, ex in var_groups:
    xl = ex + 1
    if xl < nx:
        ax_bar.plot([xl, xl], [0, ymax_bar],
                    color=group_line_color, lw=group_line_lw, zorder=5)

for s in seasons:
    for c in climates:
        y = y_pos[(s, c)]
        for x in range(nx):
            ax.add_patch(Rectangle((x, y), 1, 1,
                                   facecolor="white",
                                   edgecolor=cell_edge_color,
                                   linewidth=cell_edge_lw, zorder=0))
for y in range(ny):
    if y_to_climate.get(y) is None:
        for x in range(nx):
            ax.add_patch(Rectangle((x, y), 1, 1,
                                   facecolor="white", edgecolor="white",
                                   linewidth=0, zorder=0))

right_pad = 1.2
ax.set_xlim(0, nx + right_pad)
ax.set_ylim(-0.7, ny)
ax.set_aspect("equal")
ax_bar.set_xlim(0, nx + right_pad)

for _, sx, ex in var_groups:
    xl = ex + 1
    if xl < nx:
        ax.plot([xl, xl], [0, ny],
                color=group_line_color, lw=group_line_lw, zorder=6)

for gname, sx, ex in var_groups:
    prefix = get_var_prefix(gname)
    if prefix in type_color_map:
        ax.add_patch(Rectangle((sx, -0.55), ex - sx + 1, 0.18,
                               facecolor=type_color_map[prefix],
                               edgecolor="none", alpha=0.35, zorder=1))

ax.set_xticks(np.arange(nx) + 0.5)
ax.set_xticklabels([format_label(v) for v in variables_sorted],
                   fontsize=10, rotation=90, va="top")

yticks, ylabels = [], []
for y in range(ny):
    c = y_to_climate.get(y)
    if c:
        yticks.append(y + 0.5)
        ylabels.append(c)
ax.set_yticks(yticks)
ax.set_yticklabels(ylabels, fontsize=11)

for s in seasons:
    ax.text(nx + 0.55, season_centers[s] + 0.5, s,
            ha="center", va="center", rotation=270,
            fontsize=11, fontweight="bold", color="#2b2b2b")

for sp in ax.spines.values():
    sp.set_visible(False)
ax.tick_params(length=0)

plt.tight_layout(pad=0.4, h_pad=0.0, w_pad=0.0)
fig.subplots_adjust(hspace=0.02)

pos_bar = ax_bar.get_position()
pos_ax = ax.get_position()

ax_bar.set_position([
    pos_bar.x0,
    pos_bar.y0 - shift + gap,
    pos_bar.width,
    pos_bar.height
])
ax.set_position([
    pos_ax.x0,
    pos_ax.y0 - shift,
    pos_ax.width,
    pos_ax.height
])

rank_s, cell_pts = compute_s_values(ax, fig, RANK_FILL_RATIO)
rank_s_diamond = {rk: v * (DIAMOND_MS_SCALE ** 2) for rk, v in rank_s.items()}


def _draw_bubbles_per_cell(ax, spei_map, smrz_map, rank_s, rank_s_diamond,
                           variables_sorted, seasons, climates, y_pos):
    for xi, var in enumerate(variables_sorted):
        for s in seasons:
            for c in climates:
                cx = xi + 0.5 + OFFSET_X
                cy = y_pos[(s, c)] + 0.5 + OFFSET_Y

                items = []

                r_spei = spei_map.get((var, c, s), None)
                if r_spei in (1, 2, 3):
                    items.append({
                        "source": "spei",
                        "rank": r_spei,
                        "size": rank_s[r_spei],
                        "marker": MARKER_SPEI,
                        "color": COLOR_SPEI
                    })

                r_smrz = smrz_map.get((var, c, s), None)
                if r_smrz in (1, 2, 3):
                    items.append({
                        "source": "smrz",
                        "rank": r_smrz,
                        "size": rank_s_diamond[r_smrz],
                        "marker": MARKER_SMRZ,
                        "color": COLOR_SMRZ
                    })

                items.sort(key=lambda d: (
                    d["rank"],
                    0 if d["source"] == "spei" else 1
                ))

                base_z = 3
                for i, item in enumerate(items):
                    ax.scatter(
                        [cx], [cy],
                        s=item["size"],
                        c=item["color"],
                        marker=item["marker"],
                        edgecolors=BUBBLE_EDGE_COLOR,
                        linewidths=BUBBLE_EDGE_LW,
                        zorder=base_z + i,
                        alpha=0.92
                    )


_draw_bubbles_per_cell(ax, spei_map, smrz_map, rank_s, rank_s_diamond,
                       variables_sorted, seasons, climates, y_pos)


def draw_constrained_legend(ax_leg, ax_ref, fig, rank_s, rank_s_diamond, nx):
    fig.canvas.draw()

    ref_pos = ax_ref.get_position()
    leg_pos = ax_leg.get_position()
    ax_leg.set_position([ref_pos.x0, leg_pos.y0, ref_pos.width, leg_pos.height])

    xlim = ax_ref.get_xlim()
    ax_leg.set_xlim(xlim)
    ax_leg.set_ylim(0, 2)

    content_width = nx
    y_center = 1.0
    txt_color = "#2b2b2b"
    text_offset = 0.6

    spei_positions = np.linspace(0.5, content_width * 0.45, 4)

    ax_leg.text(spei_positions[0], y_center, "SPEI",
                ha="left", va="center", fontsize=11, fontweight="bold", color=txt_color)

    for i, rk in enumerate([1, 2, 3]):
        cx = spei_positions[i + 1]
        ax_leg.scatter([cx], [y_center], s=rank_s[rk],
                       c=COLOR_SPEI, marker=MARKER_SPEI,
                       edgecolors=BUBBLE_EDGE_COLOR, linewidths=BUBBLE_EDGE_LW,
                       zorder=10)
        ax_leg.text(cx + text_offset, y_center, f"Rank {rk}",
                    ha="left", va="center", fontsize=10, color=txt_color)

    smrz_positions = np.linspace(content_width * 0.55, content_width - 0.5, 4)

    ax_leg.text(smrz_positions[0], y_center, "SM$_{rz}$",
                ha="left", va="center", fontsize=11, fontweight="bold", color=txt_color)

    for i, rk in enumerate([1, 2, 3]):
        cx = smrz_positions[i + 1]
        ax_leg.scatter([cx], [y_center], s=rank_s_diamond[rk],
                       c=COLOR_SMRZ, marker=MARKER_SMRZ,
                       edgecolors=BUBBLE_EDGE_COLOR, linewidths=BUBBLE_EDGE_LW,
                       zorder=10)
        ax_leg.text(cx + text_offset, y_center, f"Rank {rk}",
                    ha="left", va="center", fontsize=10, color=txt_color)

    ax_leg.axis("off")


draw_constrained_legend(ax_legend, ax, fig, rank_s, rank_s_diamond, nx)

plt.savefig(out_png, dpi=600, bbox_inches="tight", facecolor="white")
plt.show()


In [ ]:
import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import matplotlib.gridspec as gridspec

file_spei = os.path.join("PATH_TO_SPEI_INPUT", "SPEI_Tsd111.xlsx")
file_smrz = os.path.join("PATH_TO_SMRZ_INPUT", "SMrz_Tsd111.xlsx")
out_png = os.path.join("PATH_TO_OUTPUT", "Combined_SPEI_SMrz_Season_Timescales_Tsd111.png")

plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 11,
    "axes.linewidth": 0.6,
    "figure.facecolor": "white",
    "axes.facecolor": "white"
})

type_color_map = {
    'VPD': "#BDBDBD",
    'SRAD': "#E6E6E6",
    'LAI': "#BDBDBD",
    'CO2': "#E6E6E6",
    'PRE': "#BDBDBD",
    'TEM': "#E6E6E6"
}

RANK_FILL_RATIO = {
    1: 0.82,
    2: 0.63,
    3: 0.44,
}

COLOR_SPEI = "#2b0076"
COLOR_SMRZ = "#A6CFCD"
MARKER_SPEI = "o"
MARKER_SMRZ = "D"
BUBBLE_EDGE_COLOR = "white"
BUBBLE_EDGE_LW = 0.5
OFFSET_X = 0.0
OFFSET_Y = 0.0

cell_edge_color = "#E6E6E6"
cell_edge_lw = 0.6
group_line_color = "#BDBDBD"
group_line_lw = 1.2

climates = ['Temperate', 'Tropical', 'Boreal', 'Arid']
seasons = ['MAM', 'JJA', 'SON', 'DJF']
gap_rows = 1

metric_order = ("mean", "trend", "cv")

bar_height_ratio = 0.3
heat_height_ratio = 6.0
legend_ratio = 0.28
bar_group_width = 0.86
bar_gap = 0
bar_single_width = (bar_group_width - bar_gap) / 2
base_color = "#E9EEF2"
show_bar_grid = False

shift = -0.14
gap = -0.165

DIAMOND_MS_SCALE = 0.72


def format_label(label: str) -> str:
    s = str(label)
    s = s.replace("_Y_", " Y_")
    s = s.replace("CO2", "CO$_2$")
    s = s.replace("SMrz", "SM$_{rz}$")
    return s


def parse_var_name(v: str):
    s = str(v).strip()
    m = re.match(r"^(?P<driver>.+?)(?P<y>_Y)?_(?P<metric>Mean|Trend|CV)$",
                 s, flags=re.IGNORECASE)
    if not m:
        return s, False, None
    return m.group("driver"), m.group("y") is not None, m.group("metric").lower()


def get_var_prefix(var_name: str):
    driver, _, _ = parse_var_name(str(var_name))
    for p in type_color_map:
        if driver.startswith(p):
            return p
    return "OTHER"


def read_rank_excel(path):
    df = pd.read_excel(path, header=[0, 1])
    variables = df.iloc[:, 0].astype(str).values
    rows = []
    for i, var in enumerate(variables):
        for j in range(1, len(df.columns)):
            v = df.iloc[i, j]
            if pd.notna(v) and str(v).strip() != "":
                try:
                    rk = int(float(v))
                    if rk in (1, 2, 3):
                        rows.append({
                            "variable": str(var),
                            "climate": str(df.columns[j][0]).strip(),
                            "season": str(df.columns[j][1]).strip(),
                            "rank": rk
                        })
                except:
                    pass
    return pd.DataFrame(rows), variables


def build_y_layout(seasons, climates, gap_rows=1):
    y_pos, y_to_climate, season_centers = {}, {}, {}
    y = 0
    for si, s in enumerate(seasons):
        start_y = y
        for c in climates:
            y_pos[(s, c)] = y
            y_to_climate[y] = c
            y += 1
        season_centers[s] = (start_y + y - 1) / 2
        if si < len(seasons) - 1:
            for _ in range(gap_rows):
                y_to_climate[y] = None
                y += 1
    return y_pos, y_to_climate, season_centers, y


def sort_and_group_variables(vars_array, metric_order=("mean", "trend", "cv")):
    vars_list = [str(v) for v in vars_array]
    driver_order, seen = [], set()
    for v in vars_list:
        d, _, _ = parse_var_name(v)
        if d not in seen:
            seen.add(d)
            driver_order.append(d)
    metric_rank = {m: i for i, m in enumerate(metric_order)}

    def key(v):
        d, iy, m = parse_var_name(v)
        return (
            driver_order.index(d) if d in driver_order else 1e9,
            1 if iy else 0,
            metric_rank.get(m, 1e6),
            v
        )

    vs = sorted(vars_list, key=key)
    groups = []
    for d in driver_order:
        idxs = [i for i, v in enumerate(vs) if parse_var_name(v)[0] == d]
        if idxs:
            groups.append((d, min(idxs), max(idxs)))
    return vs, groups


def count_ranks(variables_sorted, climates, seasons, data_map, ranks=(1, 2, 3)):
    counts = {v: {rk: 0 for rk in ranks} for v in variables_sorted}
    for v in variables_sorted:
        for s in seasons:
            for c in climates:
                r = data_map.get((v, c, s))
                if r in ranks:
                    counts[v][r] += 1
    return counts


def compute_s_values(ax, fig, rank_fill_ratio):
    fig.canvas.draw()
    ax_width_inch = ax.get_position().width * fig.get_figwidth()
    ax_width_pts = ax_width_inch * 72.0
    xlim = ax.get_xlim()
    x_data_range = xlim[1] - xlim[0]
    cell_pts = ax_width_pts / x_data_range
    s_dict = {}
    for rk, ratio in rank_fill_ratio.items():
        d_pts = ratio * cell_pts
        s_dict[rk] = np.pi * (d_pts / 2) ** 2
    return s_dict, cell_pts


df_spei, variables_spei = read_rank_excel(file_spei)
df_smrz, variables_smrz = read_rank_excel(file_smrz)

if list(map(str, variables_spei)) != list(map(str, variables_smrz)):
    raise ValueError("Variable lists are not consistent between the two tables.")

variables_sorted, var_groups = sort_and_group_variables(
    variables_spei, metric_order=metric_order
)
nx = len(variables_sorted)

y_pos, y_to_climate, season_centers, ny = build_y_layout(
    seasons, climates, gap_rows=gap_rows
)

spei_map = {(r.variable, r.climate, r.season): int(r.rank) for r in df_spei.itertuples()}
smrz_map = {(r.variable, r.climate, r.season): int(r.rank) for r in df_smrz.itertuples()}

counts_spei = count_ranks(variables_sorted, climates, seasons, spei_map)
counts_smrz = count_ranks(variables_sorted, climates, seasons, smrz_map)

fig_w = max(9.0, nx * 0.28)
fig_h = max(5.8, ny * 0.42 + 1.5)

fig = plt.figure(figsize=(fig_w, fig_h), dpi=600)

gs = gridspec.GridSpec(
    3, 1,
    height_ratios=[legend_ratio, bar_height_ratio, heat_height_ratio],
    hspace=0.0
)
ax_legend = fig.add_subplot(gs[0])
ax_bar = fig.add_subplot(gs[1])
ax = fig.add_subplot(gs[2], sharex=ax_bar)

ax_legend.axis("off")

x_centers = np.arange(nx) + 0.5
x_left = x_centers - (bar_gap / 2 + bar_single_width / 2)
x_right = x_centers + (bar_gap / 2 + bar_single_width / 2)

totals_s = np.array([sum(counts_spei[v].values()) for v in variables_sorted])
totals_m = np.array([sum(counts_smrz[v].values()) for v in variables_sorted])
ymax_bar = int(max(totals_s.max(initial=0), totals_m.max(initial=0), 1))

ax_bar.bar(x_left, np.full(nx, ymax_bar), width=bar_single_width,
           color=base_color, edgecolor="none", zorder=0)
ax_bar.bar(x_right, np.full(nx, ymax_bar), width=bar_single_width,
           color=base_color, edgecolor="none", zorder=0)

rank_alphas = {1: 1.0, 2: 0.7, 3: 0.4}
bot_l = np.zeros(nx)
bot_r = np.zeros(nx)
for rk in [1, 2, 3]:
    vl = np.array([counts_spei[v][rk] for v in variables_sorted], dtype=float)
    vr = np.array([counts_smrz[v][rk] for v in variables_sorted], dtype=float)
    ax_bar.bar(x_left, vl, bottom=bot_l, width=bar_single_width,
               color=COLOR_SPEI, alpha=rank_alphas[rk], edgecolor="none", zorder=2)
    ax_bar.bar(x_right, vr, bottom=bot_r, width=bar_single_width,
               color=COLOR_SMRZ, alpha=rank_alphas[rk], edgecolor="none", zorder=2)
    bot_l += vl
    bot_r += vr

ax_bar.set_ylim(0, ymax_bar)
ax_bar.set_ylabel("Count", fontsize=11)
ax_bar.set_yticks([0, ymax_bar])
ax_bar.set_yticklabels(["0", str(ymax_bar)], fontsize=11)
ax_bar.tick_params(axis="x", which="both", bottom=False, labelbottom=False, length=0)
ax_bar.tick_params(axis="y", length=0)

for sp in ax_bar.spines.values():
    sp.set_visible(False)
if show_bar_grid:
    ax_bar.grid(axis="y", color="#BDBDBD", lw=0.6, alpha=0.6)
for _, sx, ex in var_groups:
    xl = ex + 1
    if xl < nx:
        ax_bar.plot([xl, xl], [0, ymax_bar],
                    color=group_line_color, lw=group_line_lw, zorder=5)

for s in seasons:
    for c in climates:
        y = y_pos[(s, c)]
        for x in range(nx):
            ax.add_patch(Rectangle((x, y), 1, 1,
                                   facecolor="white",
                                   edgecolor=cell_edge_color,
                                   linewidth=cell_edge_lw, zorder=0))
for y in range(ny):
    if y_to_climate.get(y) is None:
        for x in range(nx):
            ax.add_patch(Rectangle((x, y), 1, 1,
                                   facecolor="white", edgecolor="white",
                                   linewidth=0, zorder=0))

right_pad = 1.2
ax.set_xlim(0, nx + right_pad)
ax.set_ylim(-0.7, ny)
ax.set_aspect("equal")
ax_bar.set_xlim(0, nx + right_pad)

for _, sx, ex in var_groups:
    xl = ex + 1
    if xl < nx:
        ax.plot([xl, xl], [0, ny],
                color=group_line_color, lw=group_line_lw, zorder=6)

for gname, sx, ex in var_groups:
    prefix = get_var_prefix(gname)
    if prefix in type_color_map:
        ax.add_patch(Rectangle((sx, -0.55), ex - sx + 1, 0.18,
                               facecolor=type_color_map[prefix],
                               edgecolor="none", alpha=0.35, zorder=1))

ax.set_xticks(np.arange(nx) + 0.5)
ax.set_xticklabels([format_label(v) for v in variables_sorted],
                   fontsize=10, rotation=90, va="top")

yticks, ylabels = [], []
for y in range(ny):
    c = y_to_climate.get(y)
    if c:
        yticks.append(y + 0.5)
        ylabels.append(c)
ax.set_yticks(yticks)
ax.set_yticklabels(ylabels, fontsize=11)

for s in seasons:
    ax.text(nx + 0.55, season_centers[s] + 0.5, s,
            ha="center", va="center", rotation=270,
            fontsize=11, fontweight="bold", color="#2b2b2b")

for sp in ax.spines.values():
    sp.set_visible(False)
ax.tick_params(length=0)

plt.tight_layout(pad=0.4, h_pad=0.0, w_pad=0.0)
fig.subplots_adjust(hspace=0.02)

pos_bar = ax_bar.get_position()
pos_ax = ax.get_position()

ax_bar.set_position([
    pos_bar.x0,
    pos_bar.y0 - shift + gap,
    pos_bar.width,
    pos_bar.height
])
ax.set_position([
    pos_ax.x0,
    pos_ax.y0 - shift,
    pos_ax.width,
    pos_ax.height
])

rank_s, cell_pts = compute_s_values(ax, fig, RANK_FILL_RATIO)
rank_s_diamond = {rk: v * (DIAMOND_MS_SCALE ** 2) for rk, v in rank_s.items()}


def _draw_bubbles_per_cell(ax, spei_map, smrz_map, rank_s, rank_s_diamond,
                           variables_sorted, seasons, climates, y_pos):
    for xi, var in enumerate(variables_sorted):
        for s in seasons:
            for c in climates:
                cx = xi + 0.5 + OFFSET_X
                cy = y_pos[(s, c)] + 0.5 + OFFSET_Y

                items = []

                r_spei = spei_map.get((var, c, s), None)
                if r_spei in (1, 2, 3):
                    items.append({
                        "source": "spei",
                        "rank": r_spei,
                        "size": rank_s[r_spei],
                        "marker": MARKER_SPEI,
                        "color": COLOR_SPEI
                    })

                r_smrz = smrz_map.get((var, c, s), None)
                if r_smrz in (1, 2, 3):
                    items.append({
                        "source": "smrz",
                        "rank": r_smrz,
                        "size": rank_s_diamond[r_smrz],
                        "marker": MARKER_SMRZ,
                        "color": COLOR_SMRZ
                    })

                items.sort(key=lambda d: (
                    d["rank"],
                    0 if d["source"] == "spei" else 1
                ))

                base_z = 3
                for i, item in enumerate(items):
                    ax.scatter(
                        [cx], [cy],
                        s=item["size"],
                        c=item["color"],
                        marker=item["marker"],
                        edgecolors=BUBBLE_EDGE_COLOR,
                        linewidths=BUBBLE_EDGE_LW,
                        zorder=base_z + i,
                        alpha=0.92
                    )


_draw_bubbles_per_cell(ax, spei_map, smrz_map, rank_s, rank_s_diamond,
                       variables_sorted, seasons, climates, y_pos)


def draw_constrained_legend(ax_leg, ax_ref, fig, rank_s, rank_s_diamond, nx):
    fig.canvas.draw()

    ref_pos = ax_ref.get_position()
    leg_pos = ax_leg.get_position()
    ax_leg.set_position([ref_pos.x0, leg_pos.y0, ref_pos.width, leg_pos.height])

    xlim = ax_ref.get_xlim()
    ax_leg.set_xlim(xlim)
    ax_leg.set_ylim(0, 2)

    content_width = nx
    y_center = 1.0
    txt_color = "#2b2b2b"
    text_offset = 0.6

    spei_positions = np.linspace(0.5, content_width * 0.45, 4)

    ax_leg.text(spei_positions[0], y_center, "SPEI",
                ha="left", va="center", fontsize=11, fontweight="bold", color=txt_color)

    for i, rk in enumerate([1, 2, 3]):
        cx = spei_positions[i + 1]
        ax_leg.scatter([cx], [y_center], s=rank_s[rk],
                       c=COLOR_SPEI, marker=MARKER_SPEI,
                       edgecolors=BUBBLE_EDGE_COLOR, linewidths=BUBBLE_EDGE_LW,
                       zorder=10)
        ax_leg.text(cx + text_offset, y_center, f"Rank {rk}",
                    ha="left", va="center", fontsize=10, color=txt_color)

    smrz_positions = np.linspace(content_width * 0.55, content_width - 0.5, 4)

    ax_leg.text(smrz_positions[0], y_center, "SM$_{rz}$",
                ha="left", va="center", fontsize=11, fontweight="bold", color=txt_color)

    for i, rk in enumerate([1, 2, 3]):
        cx = smrz_positions[i + 1]
        ax_leg.scatter([cx], [y_center], s=rank_s_diamond[rk],
                       c=COLOR_SMRZ, marker=MARKER_SMRZ,
                       edgecolors=BUBBLE_EDGE_COLOR, linewidths=BUBBLE_EDGE_LW,
                       zorder=10)
        ax_leg.text(cx + text_offset, y_center, f"Rank {rk}",
                    ha="left", va="center", fontsize=10, color=txt_color)

    ax_leg.axis("off")


draw_constrained_legend(ax_legend, ax, fig, rank_s, rank_s_diamond, nx)

plt.savefig(out_png, dpi=600, bbox_inches="tight", facecolor="white")
plt.show()
